# Student Performance Prediction System
## Data Preprocessing Notebook
### CodeVedX AI/ML Internship — Project 2

**Objective:**  
This notebook performs complete data preprocessing on the raw student performance dataset.  
It loads the dataset, explores its structure, handles missing values and duplicates, validates data types,  
and saves a clean, processed version ready for Exploratory Data Analysis (EDA) and Model Training.

**What this notebook covers:**
- Loading the raw CSV dataset
- Exploring dataset shape, columns, info, and statistical summary
- Checking and handling missing values
- Removing duplicate records
- Validating and cleaning invalid values
- Saving the processed dataset
- Final verification of the cleaned dataset

---
## 1. Import Libraries

**What:** Import all required Python libraries for data manipulation and analysis.  
**Why:** These libraries provide the core functionality needed to load, explore, clean, and save the dataset.  
**Expected output:** Libraries import successfully without errors.

In [ ]:
# ========================================
# STEP 1: Import Required Libraries
# ========================================

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd

print("All libraries imported successfully")

---
## 2. Configure Display Options

**What:** Configure pandas display settings for better readability.  
**Why:** Ensures all columns and rows are visible during exploration, preventing truncation of important data.  
**Expected output:** Display options configured successfully.

In [ ]:
# ========================================
# STEP 2: Configure Display Options
# ========================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print("Display options configured successfully")
print(f"  max_columns : {pd.get_option('display.max_columns')}")
print(f"  max_rows    : {pd.get_option('display.max_rows')}")

---
## 3. Setup Project Paths

**What:** Define all file paths using `pathlib.Path` with relative paths.  
**Why:** Relative paths make the notebook portable across different machines and environments.  
**Expected output:** Paths printed for verification.

In [ ]:
# ========================================
# STEP 3: Setup Project Paths
# ========================================

PROJECT_ROOT = Path("..")

# Input raw data
RAW_DATA = PROJECT_ROOT / "data" / "raw" / "StudentPerformanceFactors.csv"

# Output processed data
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed" / "student_performance.csv"

# Output directories
CHARTS_DIR = PROJECT_ROOT / "outputs" / "charts"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

# Create required directories if they do not exist
os.makedirs(PROCESSED_DATA.parent, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Project paths configured:")
print(f"  Raw Data      : {RAW_DATA}")
print(f"  Processed Data: {PROCESSED_DATA}")
print(f"  Charts        : {CHARTS_DIR}")
print(f"  Reports       : {REPORTS_DIR}")
print(f"  Models        : {MODELS_DIR}")

---
## 4. Load Dataset

**What:** Load the raw student performance CSV file into a pandas DataFrame.  
**Why:** The CSV file contains all 6,600+ records with 20 features including academic and lifestyle factors.  
**Expected output:** Dataset loaded successfully with shape displayed.

In [ ]:
# ========================================
# STEP 4: Load Dataset
# ========================================

print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = pd.read_csv(RAW_DATA)

print("\n✓ Dataset loaded successfully")
print(f"  Source: {RAW_DATA.name}")
print(f"  Rows   : {df.shape[0]}")
print(f"  Columns: {df.shape[1]}")

# Display first 5 rows
print("\nFirst 5 rows of the dataset:")
df.head()

---
## 5. Display Shape

**What:** Display the exact number of rows and columns in the dataset.  
**Why:** Understanding the dataset dimensions helps in planning preprocessing steps and memory management.  
**Expected output:** Row and column counts displayed.

In [ ]:
# ========================================
# STEP 5: Display Shape
# ========================================

print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)

print(f"\nRows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")
print(f"\nTotal cells : {df.shape[0] * df.shape[1]:,}")

---
## 6. Display Columns

**What:** List all column names in the dataset.  
**Why:** Knowing column names helps identify features, the target variable, and plan encoding strategies for categorical variables.  
**Expected output:** Complete list of 20 column names.

In [ ]:
# ========================================
# STEP 6: Display Columns
# ========================================

print("=" * 60)
print("DATASET COLUMNS")
print("=" * 60)

columns = df.columns.tolist()
print(f"\nTotal columns: {len(columns)}")
print("\nColumn names:")
for i, col in enumerate(columns, 1):
    print(f"  {i:2d}. {col}")

---
## 7. Dataset Information

**What:** Display concise summary of the DataFrame including column dtypes, non-null counts, and memory usage.  
**Why:** Reveals data types and missing values at a glance, helping identify columns that need type conversion or imputation.  
**Expected output:** Detailed info with dtypes and non-null counts.

In [ ]:
# ========================================
# STEP 7: Dataset Information
# ========================================

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")
print(df.info())

---
## 8. Statistical Summary

**What:** Generate descriptive statistics for all columns.  
**Why:** Provides insight into numerical distributions (mean, std, min, max, quartiles) and categorical value counts.  
**Expected output:** Summary statistics table.

In [ ]:
# ========================================
# STEP 8: Statistical Summary
# ========================================

print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)

print("\n--- Numerical Columns ---")
df.describe()

In [ ]:
print("--- Categorical Columns ---")
df.describe(include=["object"])

---
## 9. Check Missing Values

**What:** Count missing (null) values in each column and calculate their percentage.  
**Why:** Missing values can bias analysis and reduce model performance. Identifying them early allows proper handling.  
**Expected output:** Missing value counts and percentages per column.

In [ ]:
# ========================================
# STEP 9: Check Missing Values
# ========================================

print("=" * 60)
print("MISSING VALUES CHECK")
print("=" * 60)

missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_percent
})

# Filter only columns with missing values
missing_with_data = missing_df[missing_df["Missing Count"] > 0]

if len(missing_with_data) > 0:
    print(f"\nColumns with missing values ({len(missing_with_data)}):")
    display(missing_with_data)
else:
    print("\n✓ No missing values found in any column.")

print(f"\nTotal missing values in dataset: {df.isnull().sum().sum():,}")

---
## 10. Handle Missing Values

**What:** Apply appropriate strategies to fill or remove missing values.  
**Why:** Most ML algorithms cannot handle missing values. We need a clean dataset for modeling.  
**Strategy:**  
- Numerical columns: Fill with median (robust to outliers)  
- Categorical columns: Fill with mode (most frequent value)  
**Expected output:** Clean dataset with no missing values.

In [ ]:
# ========================================
# STEP 10: Handle Missing Values
# ========================================

print("=" * 60)
print("HANDLING MISSING VALUES")
print("=" * 60)

# Separate numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"\nNumerical columns   : {len(numerical_cols)}")
print(f"Categorical columns : {len(categorical_cols)}")

# Track how many values were filled
total_filled = 0

# Fill numerical missing values with median
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        missing_count = df[col].isnull().sum()
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled {missing_count} missing values in '{col}' with median ({median_val:.2f})")
        total_filled += missing_count

# Fill categorical missing values with mode
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        missing_count = df[col].isnull().sum()
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Filled {missing_count} missing values in '{col}' with mode ('{mode_val}')")
        total_filled += missing_count

print(f"\n✓ Missing values handled. Total values filled: {total_filled}")
print(f"\nRemaining missing values: {df.isnull().sum().sum()}")

---
## 11. Remove Duplicate Records

**What:** Identify and remove duplicate rows from the dataset.  
**Why:** Duplicates can skew analysis and lead to overfitting during model training.  
**Expected output:** Dataset with duplicates removed, count of duplicates reported.

In [ ]:
# ========================================
# STEP 11: Remove Duplicate Records
# ========================================

print("=" * 60)
print("DUPLICATE RECORDS")
print("=" * 60)

duplicate_count = df.duplicated().sum()

print(f"\nDuplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print(f"✓ Removed {duplicate_count} duplicate row(s)")
else:
    print("✓ No duplicate rows found")

print(f"\nDataset shape after removing duplicates: {df.shape}")

---
## 12. Verify Data Types

**What:** Check and confirm the data types of all columns after cleaning.  
**Why:** Ensures that numerical columns are numeric and categorical columns are strings, which is critical for downstream tasks.  
**Expected output:** Data types of all columns displayed.

In [ ]:
# ========================================
# STEP 12: Verify Data Types
# ========================================

print("=" * 60)
print("DATA TYPE VERIFICATION")
print("=" * 60)

print(f"\n{'Column':30s} {'Type':15s} {'Category':15s}")
print("-" * 60)

for col in df.columns:
    dtype = str(df[col].dtype)
    if "float" in dtype or "int" in dtype:
        category = "Numerical"
    else:
        category = "Categorical"
    print(f"{col:30s} {dtype:15s} {category:15s}")

print("\n✓ Data types verified")

---
## 13. Clean Invalid Values

**What:** Validate numerical columns for any invalid, negative, or out-of-range values.  
**Why:** Invalid values (e.g., negative Hours_Studied, Exam_Score > 100) distort analysis and model training.  
**Expected output:** Report of invalid values found and corrected.

In [ ]:
# ========================================
# STEP 13: Clean Invalid Values
# ========================================

print("=" * 60)
print("INVALID VALUE CLEANING")
print("=" * 60)

# Define validation rules for numerical columns
validation_rules = {
    "Hours_Studied": {"min": 0, "max": 168, "description": "Hours studied per week (0-168)"},
    "Attendance": {"min": 0, "max": 100, "description": "Attendance percentage (0-100)"},
    "Sleep_Hours": {"min": 0, "max": 24, "description": "Sleep hours per day (0-24)"},
    "Previous_Scores": {"min": 0, "max": 100, "description": "Previous exam scores (0-100)"},
    "Tutoring_Sessions": {"min": 0, "max": None, "description": "Number of tutoring sessions (0+)"},
    "Physical_Activity": {"min": 0, "max": None, "description": "Physical activity hours (0+)"},
    "Exam_Score": {"min": 0, "max": 100, "description": "Target exam score (0-100)"}
}

issues_found = 0
issues_corrected = 0

for col, rules in validation_rules.items():
    if col not in df.columns:
        continue
    
    # Check for negative values
    if rules["min"] is not None:
        min_violations = (df[col] < rules["min"]).sum()
        if min_violations > 0:
            print(f"  ⚠ Found {min_violations} value(s) below minimum ({rules['min']}) in '{col}'")
            df.loc[df[col] < rules["min"], col] = rules["min"]
            issues_found += min_violations
            issues_corrected += min_violations
    
    # Check for values above maximum
    if rules["max"] is not None:
        max_violations = (df[col] > rules["max"]).sum()
        if max_violations > 0:
            print(f"  ⚠ Found {max_violations} value(s) above maximum ({rules['max']}) in '{col}'")
            df.loc[df[col] > rules["max"], col] = rules["max"]
            issues_found += max_violations
            issues_corrected += max_violations

if issues_found == 0:
    print("\n✓ No invalid values found in numerical columns")
else:
    print(f"\n✓ Corrected {issues_corrected} invalid value(s)")

# Check for empty string values in categorical columns
print("\nChecking categorical columns for empty strings...")
empty_str_count = 0
for col in categorical_cols:
    empty_count = (df[col].astype(str).str.strip() == "").sum()
    if empty_count > 0:
        print(f"  ⚠ Found {empty_count} empty string(s) in '{col}'")
        df[col] = df[col].replace(r"^\s*$", np.nan, regex=True)
        df[col].fillna(df[col].mode()[0], inplace=True)
        empty_str_count += empty_count

if empty_str_count == 0:
    print("  ✓ No empty string values found")
else:
    print(f"  ✓ Filled {empty_str_count} empty string(s) with mode")

---
## 14. Save Processed Dataset

**What:** Save the cleaned and preprocessed dataset to the `data/processed/` directory.  
**Why:** The cleaned dataset will be used as input for EDA and model training notebooks,  
ensuring consistency and reproducibility across the workflow.  
**Expected output:** CSV file saved successfully.

In [ ]:
# ========================================
# STEP 14: Save Processed Dataset
# ========================================

print("=" * 60)
print("SAVING PROCESSED DATASET")
print("=" * 60)

# Ensure output directory exists
os.makedirs(PROCESSED_DATA.parent, exist_ok=True)

# Save to CSV without index
df.to_csv(PROCESSED_DATA, index=False)

print(f"\n✓ Dataset saved successfully")
print(f"  Path: {PROCESSED_DATA}")
print(f"  Rows: {df.shape[0]}")
print(f"  Columns: {df.shape[1]}")
print(f"  File size: {os.path.getsize(PROCESSED_DATA) / 1024**2:.2f} MB")

---
## 15. Reload Processed Dataset

**What:** Reload the saved CSV file into a new DataFrame to verify data integrity.  
**Why:** Confirms that the file was saved correctly and can be loaded for downstream tasks.  
**Expected output:** Dataset loaded from saved file.

In [ ]:
# ========================================
# STEP 15: Reload Processed Dataset
# ========================================

print("=" * 60)
print("RELOADING PROCESSED DATASET")
print("=" * 60)

processed_df = pd.read_csv(PROCESSED_DATA)

print(f"\n✓ Dataset reloaded successfully")
print(f"  Rows   : {processed_df.shape[0]}")
print(f"  Columns: {processed_df.shape[1]}")

print("\nFirst 5 rows:")
processed_df.head()

---
## 16. Final Verification

**What:** Perform a final comprehensive check on the cleaned dataset.  
**Why:** Ensures that all preprocessing steps were successful and the data is ready for EDA and modeling.  
**Expected output:** Final summary with no missing values, no duplicates, and correct data types.

In [ ]:
# ========================================
# STEP 16: Final Verification
# ========================================

print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print(f"\n1. Dataset Shape    : {processed_df.shape[0]} rows × {processed_df.shape[1]} columns")
print(f"2. Missing Values   : {processed_df.isnull().sum().sum()}")
print(f"3. Duplicate Rows   : {processed_df.duplicated().sum()}")
print(f"4. Memory Usage     : {processed_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n5. Column Data Types:")
for col in processed_df.columns:
    dtype = processed_df[col].dtype
    non_null = processed_df[col].notna().sum()
    print(f"    {col:30s} | {str(dtype):15s} | {non_null:>6d} non-null")

print("\n" + "=" * 60)
if processed_df.isnull().sum().sum() == 0 and processed_df.duplicated().sum() == 0:
    print("  ✓ DATA PREPROCESSING COMPLETED SUCCESSFULLY")
    print("  ✓ Dataset is clean, no missing values, no duplicates")
    print("  ✓ Ready for Exploratory Data Analysis and Model Training")
else:
    print("  ⚠ Some issues remain. Please review.")
print("=" * 60)

---
## Summary

### Tasks Completed

| Step | Task | Status |
|------|------|--------|
| 1 | Import Libraries | ✅ |
| 2 | Configure Display Options | ✅ |
| 3 | Setup Project Paths | ✅ |
| 4 | Load Dataset | ✅ |
| 5 | Display Shape | ✅ |
| 6 | Display Columns | ✅ |
| 7 | Dataset Information | ✅ |
| 8 | Statistical Summary | ✅ |
| 9 | Check Missing Values | ✅ |
| 10 | Handle Missing Values | ✅ |
| 11 | Remove Duplicates | ✅ |
| 12 | Verify Data Types | ✅ |
| 13 | Clean Invalid Values | ✅ |
| 14 | Save Processed Dataset | ✅ |
| 15 | Reload Processed Dataset | ✅ |
| 16 | Final Verification | ✅ |

### Output
- **Processed Dataset:** `data/processed/student_performance.csv`

### Next Steps
- Proceed to **Notebook 2: Exploratory Data Analysis (EDA)**